# LIME explanations - step 1: a few test images

**LIME** (Ribeiro et al., 2016) explains one prediction of a trained model: which parts of the image did the
model rely on? It splits the image into **superpixels** (SLIC), hides random subsets of them in 1000 copies of
the image, asks the model about every copy, and fits a weighted linear model from "which superpixels are
visible" to "probability of the class". The weight of a superpixel is its **importance**.
The LIME code is our own (`src/explain.py`); SLIC comes from scikit-image.

**Step 1 (this notebook):** the best model, ConvNeXt-Tiny, on 8 test images - 4 wrong predictions and, for each,
a correct one of the same material from another scene. For a wrong prediction both the predicted and the true
class are explained. Only figures and a small table; no measurements yet.

* The model is the analysed run of the training notebook (best validation accuracy), weights from `outputs/`
  or Google Drive.
* The images are the test images exactly as the model sees them (whole image, 512x512).
* The bounding boxes are not used.
* Outputs: `outputs/lime/<model>/`, saved to `results/lime/<model>/` on GitHub (and Google Drive).

**Run on a GPU** (university computer or Colab): about 2 seconds per explanation. On a CPU about 10 minutes.

## 1. Setup: code and data

In [ ]:
# Works in Colab and on an own computer with an NVIDIA GPU (e.g. the university computer).
import os, sys
REPO_URL = "https://github.com/Ahmed-AlTahan/CNN-LIME.git"

if "google.colab" in sys.modules:          # Colab: get the code from GitHub
    REPO_DIR = "/content/thesis"
    if not os.path.isdir(REPO_DIR):
        !git clone -q {REPO_URL} {REPO_DIR}
else:                                      # own computer: this notebook is in <repository>/notebooks/
    REPO_DIR = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
!git -C "{REPO_DIR}" pull -q --ff-only
os.chdir(REPO_DIR)
sys.path.insert(0, os.path.join(REPO_DIR, "src"))

import env
env.ensure_dataset()      # downloads the dataset into data/ the first time
env.print_info()

## 2. Settings
The LIME settings come from `config.LIME`; change them here for an experiment.

In [ ]:
import config, explain, env

MODEL = "convnext_tiny"          # the model to explain (another trained model: "resnet50", ...)
settings = config.LIME
# e.g.: settings = config.replace(config.LIME, n_segments=400, n_samples=2000)
out_dir = explain.output_dir(MODEL)
print(settings)

## 3. Load the trained model
The run with the best validation accuracy, the same one analysed in the training notebook. If its weights are not on this computer, they are taken from Google Drive.

In [ ]:
model, seed = explain.load_trained(MODEL)

## 4. Choose the test images
From the test predictions of that run: 4 wrong predictions (metal confused with metal first, then black plastic), each paired with a correct image of the same material from another scene.

In [ ]:
examples = explain.choose_examples(MODEL)

## 5. Explain them
One figure per image:
* **input**: the image as the model sees it, **superpixels**: the SLIC regions LIME switches on and off
* **top 5 superpixels**: the regions that support the explained class most (everything else is faded)
* **importance**: red = supports the class, blue = speaks against it
* **p**: the model's probability for that class; **fidelity R2**: how well LIME's linear model imitates the
  network around this image (close to 1 = the explanation is trustworthy)

For a wrong prediction the second row explains the **true** class: what would have spoken for the right answer.

In [ ]:
rows = explain.explain_examples(model, examples, settings, out_dir, MODEL)

## 6. Save the results
Figures and the table go to `results/lime/<model>/` on GitHub (and to `MyDrive/thesis_results/lime/<model>/` on Google Drive). A re-run replaces them.

In [ ]:
env.save_results(out_dir)